# ACER v3 — Experimental Analysis

This notebook is the analysis and visualization layer for ACER.

Keep core compliance logic in Python modules. Use the notebook for:
- reproducible experiments
- tabular analysis
- metrics
- visualizations
- interpretation

Future comparisons:
- manual requirement engineering
- LLM-assisted requirement engineering
- MBSE
- LLM + MBSE
- LLM + MBSE + agentic adaptation


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

from app.engine import assess
from app.loaders import load_system, load_requirements
from app.adaptation import load_tactics
from app.agents import ComplianceOrchestrator


In [ ]:
reqs = load_requirements(ROOT / "data/requirements/demo_requirements.yaml")
tactics = load_tactics(ROOT / "data/tactics/tactics.yaml")
system = load_system(ROOT / "data/systems/recruitment_noncompliant.yaml")

baseline = assess(system, reqs)

pd.DataFrame([
    {
        "requirement": r.requirement_id,
        "status": r.status,
        "explanation": r.explanation,
    }
    for r in baseline.results
])


In [ ]:
from app.scenarios import write_scenarios

generated = ROOT / "data/systems/generated"
write_scenarios(generated, n=30, seed=42)

rows = []
for path in sorted(generated.glob("scenario-*.yaml")):
    s = load_system(path)
    report = assess(s, reqs)
    for r in report.results:
        rows.append({
            "system_id": s.id,
            "requirement_id": r.requirement_id,
            "status": r.status,
        })

scenario_df = pd.DataFrame(rows)
scenario_df.head()


In [ ]:
baseline_counts = (
    scenario_df.groupby("status")
    .size()
    .reindex(["PASS", "FAIL", "UNCERTAIN"], fill_value=0)
)

baseline_counts


In [ ]:
ax = baseline_counts.plot(kind="bar", figsize=(7, 4), legend=False)
ax.set_title("Baseline compliance outcomes")
ax.set_xlabel("Status")
ax.set_ylabel("Requirement checks")
plt.tight_layout()
plt.show()


In [ ]:
adapt_rows = []

for path in sorted(generated.glob("scenario-*.yaml")):
    s = load_system(path)
    before = assess(s, reqs)
    final_system, history = ComplianceOrchestrator(reqs, tactics).run(s)
    after = assess(final_system, reqs)

    adapt_rows.append({
        "system_id": s.id,
        "before": before.overall_status,
        "after": after.overall_status,
        "adaptation_steps": len(history),
        "adaptation_success": after.overall_status == "COMPLIANT",
    })

adapt_df = pd.DataFrame(adapt_rows)
adapt_df


In [ ]:
print("Adaptation success rate:",
      round(100 * adapt_df["adaptation_success"].mean(), 2), "%")

step_counts = adapt_df["adaptation_steps"].value_counts().sort_index()
ax = step_counts.plot(kind="bar", figsize=(7, 4), legend=False)
ax.set_title("Adaptation steps per scenario")
ax.set_xlabel("Number of steps")
ax.set_ylabel("Systems")
plt.tight_layout()
plt.show()


## Research extension

The production version of this notebook should additionally calculate:

1. Requirement-extraction accuracy
2. Compliance precision / recall / F1
3. False-positive and false-negative rates
4. Adaptation success and number of steps
5. Adaptation cost/disruption
6. Evidence completeness
7. Explanation quality
8. Human-intervention frequency
9. Uncertainty rate
10. Statistical comparison of alternative engineering approaches

Raw experiment output should be stored under `data/results/`; plots and
interpretation should remain in the notebook.
